# L9d Lab: Estimating a Logistic Regression Binary Classifier using Gradient Descent

In this lab, we will implement a logistic regression binary classifier using gradient descent. Logistic regression is a statistical method for predicting binary outcomes based on one or more predictor variables. Unlike the Perceptron, it gives each prediction a probability, which tells us how sure the classifier is about each label.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
>
> * __Implement gradient descent for logistic regression:__ Gradient descent estimates the parameters by repeatedly stepping against the gradient of the regularized cross-entropy loss. We'll complete the training loop, fit a classifier to the banknote data, and watch the loss fall.
> * __Compare logistic regression with the Perceptron:__ Both classifiers separate the two classes with a hyperplane, but they choose the hyperplane in different ways. We'll train both on the same banknotes and compare their testing mistakes and how much their answers change when we train a little longer.
> * __Use the predicted probabilities to make decisions:__ The probability of each label tells us which banknotes the classifier is unsure about. We'll send the uncertain banknotes to a human inspector and see how the regularization parameter changes the number we send.

In [the L9c lecture](../L9c/CHEME-5800-L9c-Lecture-LogisticRegressionAndOptimization-Fall-2026.ipynb), we derived logistic regression and estimated its parameters with gradient descent. Here, we apply it to the banknote dataset from [the L9a example](../L9a/CHEME-5800-L9a-Example-LinearModels-Classification-Perceptron-Fall-2026.ipynb), with the same training and testing banknotes, so we can compare it with the Perceptron.

Let's get started!
___

## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course package, this lab's code in the [src](src) folder, and the packages used here.

Let's set up our code environment:

In [ ]:
# Load packages and paths from this notebook's local setup file -
include(joinpath(@__DIR__, "Include.jl"));

# Choose one theme; after changing it, rerun this cell and the plotting cells -
theme(:default)   # light background
# theme(:dark)   # dark background

In addition to standard Julia libraries, we'll use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material.

This lab's plotting function is in [src/Visualize.jl](src/Visualize.jl). Your gradient descent loop goes in [src/Compute.jl](src/Compute.jl). We call [the `L9dLogistic.my_logistic_regression(...)` function](src/Compute.jl) by its qualified name, so that re-running the setup cell picks up your edits.

### Data
We use the [banknote authentication dataset from the UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/267/banknote+authentication), which we explored in the L9a example. It holds 1,372 banknotes, each described by four features computed from an image of the note. The class is $y\in\{-1,1\}$, where $-1$ indicates genuine and $1$ indicates forged.

[The `MyBanknoteAuthenticationDataset(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/data/#VLDataScienceMachineLearningPackage.MyBanknoteAuthenticationDataset) loads the dataset [as a `DataFrame` instance](https://dataframes.juliadata.org/stable/). We split it into the data matrix $\mathbf{X}$ (every column except `class`) and the label vector $\mathbf{y}$ (the `class` column):

In [ ]:
df_banknote = MyBanknoteAuthenticationDataset(); # load the banknote dataset as a DataFrame
X = Matrix(df_banknote[:, Not(:class)]); # data matrix: select all the columns *except* class
y = Vector(df_banknote[:, :class]); # label vector: select the class column

Next, let's partition the data into a `training` set and a `testing` set. We use the same seeded random split as the L9a example, so both notebooks use the same 1,097 training and 275 testing banknotes.

__Feature scaling__: The standard deviation of `skewness` is almost three times that of `entropy`. The penalty in the regularized loss treats every parameter the same, so we __standardize__ each feature: we subtract its mean and divide by its standard deviation, both computed from the training banknotes only.

Finally, we append a column of ones for the bias, which gives the augmented features $\hat{\mathbf{x}}$. Each set is a `NamedTuple` with the fields `X` (the augmented, scaled data matrix) and `y` (the labels):

In [ ]:
training, testing = let

    # initialize -
    s = 0.80; # fraction of data for training
    seed = 1234; # seed for the random number generator: the same split as the L9a example
    rng = MersenneTwister(seed); # random number generator used only for this split
    number_of_samples = size(X,1); # number of banknotes in the dataset
    number_of_training_samples = floor(Int, s*number_of_samples); # 80% of the data for training
    i = randperm(rng, number_of_samples); # random permutation of the row indices
    training_indices = i[1:number_of_training_samples]; # first 80% of the shuffled indices
    testing_indices = i[number_of_training_samples+1:end]; # last 20% of the shuffled indices

    # standardize, with the mean and standard deviation of each feature from the training banknotes only -
    feature_mean = mean(X[training_indices, :], dims=1); # 1×4 row: the mean of each feature
    feature_std = std(X[training_indices, :], dims=1); # 1×4 row: the standard deviation of each feature
    Z = (X .- feature_mean) ./ feature_std; # scaled features, every banknote (fancy, what is going on here?)

    # setup training -
    one_vector = ones(number_of_training_samples); # column of ones for the bias
    training = (X=[Z[training_indices, :] one_vector], y=y[training_indices]);

    # setup testing -
    one_vector = ones(length(testing_indices)); # column of ones for the bias
    testing = (X=[Z[testing_indices, :] one_vector], y=y[testing_indices]);

    # print the size of each set -
    println("training: ", length(training.y), " banknotes, testing: ", length(testing.y), " banknotes")

    training, testing; # return
end;

___

## Task 1: Estimate the Parameters with Gradient Descent

In this task, we write the gradient descent loop for logistic regression and use it to estimate the parameters $\mathbf{\theta}$ from the training banknotes. From the lecture, the regularized cross-entropy loss and its gradient are given by:
$$
\begin{align*}
J_{\delta}(\mathbf{\theta}) & = \sum_{i=1}^{n}\log\!\bigl(1+e^{-u_{i}}\bigr) + \frac{\delta}{2}\,\lVert\mathbf{\theta}\rVert_{2}^{2}\\
\nabla J_{\delta}(\mathbf{\theta}) & = -2\beta\sum_{i=1}^{n}\bigl(1-\sigma(u_{i})\bigr)\,y_{i}\,\hat{\mathbf{x}}_{i} + \delta\,\mathbf{\theta}
\end{align*}
$$
where the sums run over the $n$ training banknotes, $u_{i} = 2\beta y_{i}\,\hat{\mathbf{x}}_{i}^{\top}\mathbf{\theta}$, $\sigma(z)=1/(1+e^{-z})$ is the logistic function, $\beta$ is the inverse temperature, and $\delta\geq 0$ is the regularization parameter. We fix $\beta=1$ and start with no penalty, $\delta=0$.

We use the constant learning rate $\alpha = 1/(L+\delta)$, where $L=\beta^{2}\lVert\hat{\mathbf{X}}\rVert_{2}^{2}$ and $\lVert\hat{\mathbf{X}}\rVert_{2}$ is the largest singular value of the training data matrix (the `training.X` array). With this learning rate, no step increases the loss. Let's set the constants and define the loss $J_{\delta}(\mathbf{\theta})$ as a one-line function, which we use in the checks below:

In [ ]:
β = 1.0; # inverse temperature: we fix it and learn θ
δ = 0.0; # regularization parameter: no penalty for now
ϵ = 1e-6; # stopping tolerance on the change in θ
maxiter = 50_000; # maximum number of gradient descent steps
L = β^2*opnorm(training.X)^2; # β² times the largest singular value of training.X, squared
α = 1/(L + δ); # learning rate: the largest constant step that the lecture's bound allows
J(θ, δ) = sum(log(1 + exp(-2β*training.y[i]*dot(training.X[i,:], θ))) for i ∈ eachindex(training.y)) + (δ/2)*dot(θ, θ); # regularized loss on the training data
println("L = ", round(L, digits=1), ", learning rate α = ", round(α, sigdigits=4))

### Implement Gradient Descent

Now let's write the gradient descent loop ourselves, following the L9c algorithm. Open [`src/Compute.jl`](src/Compute.jl) and complete [the `my_logistic_regression(...)` function](src/Compute.jl). Everything except three lines is already written, and a TODO comment marks each of them:

> __Implementation tasks__
>
> * __TODO 1:__ The gradient. Inside the loop over the training examples, replace the right-hand side of `∇J = ∇J` with `∇J` plus term $i$ of the gradient, $-2\beta\bigl(1-\sigma(u_{i})\bigr)\,y_{i}\,\hat{\mathbf{x}}_{i}$. The penalty term $\delta\,\mathbf{\theta}$ is already in `∇J` when the loop starts.
> * __TODO 2:__ The update. Replace the right-hand side of `θ_new = θ` with $\mathbf{\theta}-\alpha\,\nabla J_{\delta}(\mathbf{\theta})$.
> * __TODO 3:__ The stopping test. Replace `false` with step 3 of the L9c algorithm: this step changed $\mathbf{\theta}$ by at most $\epsilon$, or $k$ has reached $\texttt{maxiter}$.

Save the file and re-run the setup cell at the top of the notebook to reload your edits, then run the cell below. Until your loop returns a result, the call throws an error.

We save the returned `NamedTuple` in the `result` variable. Its fields are the parameters `θ`, the number of `iterations`, a `converged` flag, and the `losses`, before the first step and after each step:

In [ ]:
result = L9dLogistic.my_logistic_regression(training.X, training.y,
    α = α, β = β, δ = δ, ϵ = ϵ, maxiter = maxiter); # the untouched stub throws an error
println("iterations: ", result.iterations, ", converged: ", result.converged, ", final loss: ", round(result.losses[end], digits=3))

### Check: Your Gradient Descent

With our learning rate, the loss should never increase. We also run your function with a strong penalty, $\delta=10$, which converges quickly. At its answer $\hat{\mathbf{\theta}}$, the gradient of $J_{10}$ should be close to zero. We estimate the gradient from loss values alone, with a [central finite difference](https://en.wikipedia.org/wiki/Finite_difference) in each parameter, so the check does not depend on your gradient code:

In [ ]:
@testset "My logistic regression" begin

    # the loss never increases (diff gives the change from each loss to the next) -
    @test all(diff(result.losses) .≤ 1e-9)

    # with δ = 0, the loop uses every step without meeting the stopping test -
    @test result.iterations == maxiter && result.converged == false

    # with δ = 10, it meets the stopping test well before maxiter -
    strong = L9dLogistic.my_logistic_regression(training.X, training.y,
        α = 1/(L + 10.0), β = β, δ = 10.0, ϵ = ϵ, maxiter = maxiter);
    @test strong.converged && strong.iterations < maxiter

    # ... and the finite-difference gradient of the loss at its answer is close to zero -
    h = 1e-5; # finite-difference step
    ∇J_fd = zeros(length(strong.θ)); # one entry per parameter
    for j ∈ eachindex(strong.θ)
        e = zeros(length(strong.θ)); # unit vector in direction j
        e[j] = 1.0;
        ∇J_fd[j] = (J(strong.θ + h*e, 10.0) - J(strong.θ - h*e, 10.0))/(2h); # central difference
    end
    @test norm(∇J_fd) < 1e-2
end;

Let's plot the loss after each step. We use a logarithmic axis for the iteration count, so both the first few steps and the last few thousand show up:

In [ ]:
let

    # initialize -
    k = 1:result.iterations; # iteration counts
    losses = result.losses[2:end]; # the loss after each step

    # plot the loss against the iteration count -
    plot(k, losses, xscale = :log10, lw = 2, label = "δ = $(δ)", framestyle = :box,
        xlabel = "Iteration k", ylabel = "Regularized loss")
end

__What do we see?__ With $\mathbf{\theta}=\mathbf{0}$, every probability is $1/2$, so the loss starts at $n\log 2\approx 760$. It falls below 500 in the first step and to about 33 after a thousand steps, then creeps down to about 20.4. Gradient descent used all 50,000 steps without meeting the stopping test.

### Things to think about

__Question:__ Why does gradient descent need so many steps on the banknotes, when the L9c example converged in a few hundred? What would you expect to happen to the number of steps if we added a penalty, $\delta>0$? Briefly explain.

___

## Task 2: Compare Logistic Regression with the Perceptron

In this task, we classify the testing banknotes with our logistic regression model and with the Perceptron, trained on the same scaled training banknotes, and compare their mistakes.

__Inference__: For each testing banknote, the model gives the probability that it is forged, $P_{\hat{\mathbf{\theta}}}(y=1\mid\hat{\mathbf{x}})=\sigma(2\beta\,\hat{\mathbf{x}}^{\top}\hat{\mathbf{\theta}})$. We predict forged, $\hat{y}=1$, when this probability is at least $1/2$, and genuine, $\hat{y}=-1$, otherwise. We store the probabilities in the `P_logistic::Vector{Float64}` variable and the predicted labels in the `ŷ_logistic::Vector{Int64}` variable:

In [ ]:
P_logistic, ŷ_logistic = let

    # initialize -
    X = testing.X; # augmented testing data matrix
    θ = result.θ; # estimated parameters: four weights, then the bias
    number_of_examples = size(X,1); # number of testing banknotes
    P = zeros(number_of_examples); # probability that each testing banknote is forged
    ŷ = zeros(Int64, number_of_examples); # predicted label of each testing banknote

    # compute the probability and the label for each testing banknote -
    for i ∈ 1:number_of_examples
        P[i] = 1/(1 + exp(-2β*dot(X[i,:], θ))); # σ(2β x̂ᵢᵀθ)
        ŷ[i] = P[i] ≥ 0.5 ? 1 : -1; # forged when the probability is at least 1/2
    end

    P, ŷ # return
end;

Next, let's train the Perceptron as in the L9a example, from $\mathbf{\theta}=\mathbf{0}$ with the mistake threshold $M=0$ and $T=1000$ passes, but on the scaled features. [The `learn(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/binaryclassification/#VLDataScienceMachineLearningPackage.learn) trains it, and [the `classify(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/binaryclassification/#VLDataScienceMachineLearningPackage.classify) returns the sign of each testing banknote's score. We store the predicted labels in the `ŷ_perceptron::Vector{Float64}` variable:

In [ ]:
ŷ_perceptron = let

    # initialize -
    T = 1000; # maximum number of passes through the training data
    model = build(MyPerceptronClassificationModel, (
        parameters = zeros(size(training.X,2)), # start from θ = 0
        mistakes = 0 # mistake threshold M
    ));

    # train, then classify the testing banknotes -
    model = learn(training.X, training.y, model, maxiter = T); # the learned parameters are in model.β
    ŷ = classify(testing.X, model); # the sign of each score
    ŷ[ŷ .== 0] .= 1; # a score of exactly zero gets the label 1, as in L9a

    ŷ # return
end;

Let's compute the confusion matrix for each classifier with [the `confusion(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/binaryclassification/#VLDataScienceMachineLearningPackage.confusion) and compare the counts. Forged is the positive class, so a false negative (FN) is a forgery that gets through, and a false positive (FP) flags a genuine banknote as forged.

__Prediction:__ Which classifier will make fewer mistakes on the testing banknotes: {logistic regression, the Perceptron, or neither}?

In [ ]:
let

    # initialize -
    df = DataFrame(); # one row per classifier
    predictions = [("logistic regression", ŷ_logistic), ("Perceptron", ŷ_perceptron)]; # (name, predicted labels)

    # compute the confusion matrix and the mistakes for each classifier -
    for (name, ŷ) ∈ predictions
        CM = confusion(testing.y, ŷ); # important: actual labels first, predicted labels second
        TP = CM[1,1]; # forged banknotes labeled forged
        FN = CM[1,2]; # forged banknotes labeled genuine (forgeries that get through)
        FP = CM[2,1]; # genuine banknotes labeled forged (false alarms)
        TN = CM[2,2]; # genuine banknotes labeled genuine
        push!(df, (classifier = name, TP = TP, FN = FN, FP = FP, TN = TN, mistakes = FN + FP,
            accuracy = round((TP + TN)/(TP + TN + FP + FN), digits=3)));
    end

    # show the table -
    pretty_table(df;
        show_first_column_label_only = true,
        display_size = (-1, -1), # show every row and column
        alignment = [:l, :r, :r, :r, :r, :r, :r] # left-align text and right-align numerical columns
    );
end

__What do we see?__ With our seed and split, logistic regression made 2 mistakes on the 275 testing banknotes, both forgeries that got through, and the Perceptron made 5; your counts may differ if you change the seed. (On the raw features in the L9a example, the Perceptron made 12; scaling changes every Perceptron update.)

No line separates these banknotes, so the Perceptron changes its parameters in every pass, and its answer depends on the pass it stops after. Gradient descent instead lowers the loss with every step.

__Prediction:__ Suppose we let the Perceptron make one more pass, $T=1001$. Will it make {more, fewer, or the same number of} mistakes on the testing banknotes? What about one more gradient descent step?

In [ ]:
let

    # one more Perceptron pass, T = 1001 -
    model = build(MyPerceptronClassificationModel, (parameters = zeros(size(training.X,2)), mistakes = 0));
    model = learn(training.X, training.y, model, maxiter = 1001); # one more pass than before
    ŷ = classify(testing.X, model); # the sign of each score
    ŷ[ŷ .== 0] .= 1; # a score of exactly zero gets the label 1
    println("Perceptron with T = 1001: ", count(ŷ .!= testing.y), " testing mistakes")

    # one more gradient descent step, maxiter + 1 -
    longer = L9dLogistic.my_logistic_regression(training.X, training.y,
        α = α, β = β, δ = δ, ϵ = ϵ, maxiter = maxiter + 1);
    ŷ = [dot(testing.X[i,:], longer.θ) ≥ 0 ? 1 : -1 for i ∈ 1:size(testing.X,1)]; # σ(2βz) ≥ 1/2 exactly when z ≥ 0
    println("logistic regression with one more step: ", count(ŷ .!= testing.y), " testing mistakes; θ moved by ",
        round(norm(longer.θ - result.θ), sigdigits=2))
end

__What do we see?__ One more pass changed the Perceptron's testing mistakes from 5 to 7. One more gradient descent step moved $\mathbf{\theta}$ by less than $10^{-4}$ and left the logistic regression labels unchanged.

Logistic regression also tells us how sure it is about each label. [The `plot_probabilities(...)` function](src/Visualize.jl) plots the probability that each testing banknote is forged, with one row for each actual class (forged in orange, genuine in teal). It colors the mistakes gray and draws the threshold $1/2$ as a dashed line:

In [ ]:
plot_probabilities(P_logistic, testing.y, title = "Logistic regression, δ = $(δ)")

__What do we see?__ Most testing banknotes sit near 0 or near 1: 254 of the 275 have a probability below 0.01 or above 0.99. The two missed forgeries have probabilities of about 0.29 and 0.33, below the threshold but far from 0, so the model was unsure about them.

### Things to think about

__Question:__ Why does one extra pass change the Perceptron's answer, while one extra gradient descent step does not? Would stopping gradient descent much earlier, say after 10,000 steps, change its labels? Briefly explain.

___

## Task 3: Send the Uncertain Banknotes to an Inspector

In this task, we use the probabilities to decide which banknotes a person should check. Suppose a bank accepts or rejects each banknote automatically when the classifier is sure, and sends the rest to a human inspector.

We pick an __inspector band__ $(t_{\ell}, t_{h})$, with $t_{\ell}<1/2<t_{h}$. A banknote whose probability of being forged is at most $t_{\ell}$ is accepted as genuine, one whose probability is at least $t_{h}$ is rejected as forged, and one in between goes to the inspector. Let's start with the band $(0.25, 0.75)$:

In [ ]:
t_low = 0.25; # accept as genuine when the probability of forged is at most t_low
t_high = 0.75; # reject as forged when the probability of forged is at least t_high

For each testing banknote, we check whether its probability lies in the band, and we count the mistakes among the banknotes decided automatically. Outside the band, the automatic decision is the label at the threshold $1/2$, so we compare `ŷ_logistic` with the actual labels.

__Prediction:__ Will {none, a few, or most} of the 275 testing banknotes go to the inspector, and will {zero or some} mistakes be left among the banknotes decided automatically?

In [ ]:
let

    # initialize -
    inspect = findall(p -> t_low < p < t_high, P_logistic); # banknotes in the band go to the inspector
    automatic = findall(p -> p ≤ t_low || p ≥ t_high, P_logistic); # the rest are decided automatically

    # count the mistakes among the automatic decisions -
    mistakes = count(ŷ_logistic[automatic] .!= testing.y[automatic]);

    println("band (", t_low, ", ", t_high, "): ", length(inspect), " banknotes to the inspector, ",
        length(automatic), " decided automatically, with ", mistakes, " mistakes")
end

Let's add the band to the probability figure:

In [ ]:
plot_probabilities(P_logistic, testing.y, band = (t_low, t_high), title = "Inspector band ($(t_low), $(t_high)), δ = $(δ)")

__What do we see?__ Five banknotes land in the band, including both missed forgeries. The inspector checks those 5, and the classifier decides the other 270 without a mistake.

__Prediction:__ What happens if we narrow the band to $(0.4, 0.6)$? Do {more, fewer, or the same number of} banknotes go to the inspector, and are {zero or some} automatic mistakes left?

In [ ]:
let

    # a narrower band -
    band = (0.4, 0.6); # (t_low, t_high)
    inspect = findall(p -> band[1] < p < band[2], P_logistic); # banknotes in the band go to the inspector
    automatic = findall(p -> p ≤ band[1] || p ≥ band[2], P_logistic); # the rest are decided automatically
    mistakes = count(ŷ_logistic[automatic] .!= testing.y[automatic]); # mistakes among the automatic decisions
    println("band ", band, ": ", length(inspect), " banknotes to the inspector, ", mistakes, " automatic mistakes")

    # show the band -
    plot_probabilities(P_logistic, testing.y, band = band, title = "Inspector band $(band), δ = $(δ)")
end

__What do we see?__ With the band $(0.4, 0.6)$, only 2 genuine banknotes go to the inspector. Both missed forgeries fall outside the band and are accepted as genuine automatically, so 2 mistakes are left.

Finally, let's see how the penalty changes the probabilities.

__Prediction:__ Suppose we refit the model with $\delta=10$. Will gradient descent need {more or fewer} steps, and will {more, fewer, or the same number of} banknotes go to the inspector with the band $(0.25, 0.75)$?

In [ ]:
P_strong = let

    # refit with a strong penalty -
    δ_strong = 10.0; # regularization parameter
    fit = L9dLogistic.my_logistic_regression(training.X, training.y,
        α = 1/(L + δ_strong), β = β, δ = δ_strong, ϵ = ϵ, maxiter = maxiter);
    println("δ = ", δ_strong, ": ", fit.iterations, " iterations, converged: ", fit.converged)

    # probabilities and labels for the testing banknotes, as in Task 2 but written as comprehensions -
    P = [1/(1 + exp(-2β*dot(testing.X[i,:], fit.θ))) for i ∈ 1:size(testing.X,1)]; # σ(2β x̂ᵢᵀθ)
    ŷ = [p ≥ 0.5 ? 1 : -1 for p ∈ P]; # forged when the probability is at least 1/2
    println("testing mistakes at the threshold 1/2: ", count(ŷ .!= testing.y))

    # the inspector band (t_low, t_high) -
    inspect = findall(p -> t_low < p < t_high, P); # banknotes in the band go to the inspector
    automatic = findall(p -> p ≤ t_low || p ≥ t_high, P); # the rest are decided automatically
    println("band (", t_low, ", ", t_high, "): ", length(inspect), " banknotes to the inspector, ",
        count(ŷ[automatic] .!= testing.y[automatic]), " automatic mistakes")

    P # return
end;

In [ ]:
plot_probabilities(P_strong, testing.y, band = (t_low, t_high), title = "Inspector band ($(t_low), $(t_high)), δ = 10")

__What do we see?__ With $\delta=10$, gradient descent converged in under 700 steps, but the model makes 6 testing mistakes at the threshold $1/2$ instead of 2. Fewer probabilities sit near 0 or 1: 76 of the 275 are below 0.01 or above 0.99, compared with 254 before.

Now 12 banknotes land in the band instead of 5, and all 6 mistakes are among them, so again no automatic mistake is left.

### Things to think about

__Question:__ Which band would you choose if an inspector's time is cheap? What if letting a forgery through is very costly? Why did the larger penalty send more banknotes to the inspector here? Briefly explain.

___

## Summary

In this lab, we estimated a logistic regression classifier for the banknotes with gradient descent, compared it with the Perceptron, and used its probabilities to decide which banknotes to check by hand.

> __Key Takeaways:__
>
> * __Gradient descent for logistic regression:__ We completed a gradient descent loop for the regularized cross-entropy loss and checked that the loss never increased. Without a penalty, the loss was still falling slowly after 50,000 steps, while a strong penalty let the same loop converge in under 700.
>
> * __Logistic regression and the Perceptron on the same banknotes:__ We trained both classifiers on the same scaled training banknotes, and with our split, logistic regression made fewer testing mistakes than the Perceptron. One more Perceptron pass changed its number of mistakes, while one more gradient descent step left the logistic regression labels unchanged.
>
> * __Probabilities support decisions:__ We sent the banknotes with a probability in the middle of the range to an inspector, and with our band every testing mistake was among them, so no mistake was left among the automatic decisions. With a strong penalty, fewer probabilities were near 0 or 1, and more banknotes went to the inspector.

We can now change the seed of the split, the penalty, or the inspector band and see how the mistakes and the number of checks by hand respond.

___